# 20minutos



In [20]:
!pip -q install requests beautifulsoup4 pandas lxml

import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import re
import time
import json

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:149.0) Gecko/20100101 Firefox/149.0"
}

FECHA_INICIO = datetime(2026, 1, 1)
FECHA_FIN = datetime(2026, 4, 21, 23, 59, 59)

BASES_TEMA = [
    "https://www.20minutos.es/tags/temas/estrecho-ormuz.html",
    "https://www.20minutos.es/tags/temas/conflicto-iran-eeuu.html",
    "https://www.20minutos.es/tags/temas/guerra-israel-iran.html",
    "https://www.20minutos.es/tags/lugares/iran.html",
    "https://www.20minutos.es/tags/lugares/golfo-persico.html",
]

KEYWORDS_ORMUZ = ["ormuz", "estrecho de ormuz"]

def normalizar_texto(texto):
    return re.sub(r"\s+", " ", str(texto or "")).strip()

def contiene_ormuz(texto):
    t = normalizar_texto(texto).lower()
    return any(k in t for k in KEYWORDS_ORMUZ)

def parsear_fecha_visible_20minutos(fecha_str):
    meses = {
        "ene": 1, "feb": 2, "mar": 3, "abr": 4,
        "may": 5, "jun": 6, "jul": 7, "ago": 8,
        "sep": 9, "oct": 10, "nov": 11, "dic": 12
    }
    texto = normalizar_texto(fecha_str).lower()
    m = re.search(r"(\d{1,2})\s+([a-záéíóú]{3})\s+(\d{4})\s*-\s*(\d{1,2}):(\d{2})", texto)
    if not m:
        return None
    dia = int(m.group(1))
    mes_txt = m.group(2)[:3]
    anio = int(m.group(3))
    hora = int(m.group(4))
    minuto = int(m.group(5))
    mes = meses.get(mes_txt)
    if not mes:
        return None
    try:
        return datetime(anio, mes, dia, hora, minuto)
    except:
        return None

def parsear_fecha_iso(fecha_str):
    if not fecha_str:
        return None
    try:
        return datetime.fromisoformat(str(fecha_str).replace("Z", "+00:00")).replace(tzinfo=None)
    except:
        return None

def esta_en_rango(dt):
    return dt is not None and FECHA_INICIO <= dt <= FECHA_FIN

def construir_paginas(base_url, max_paginas=8):
    paginas = [base_url]
    base = base_url.rstrip("/")
    for i in range(2, max_paginas + 1):
        paginas.append(f"{base}/{i}/")
    return paginas

def sacar_enlaces_desde_pagina_tema(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        enlaces = []
        for a in soup.find_all("a", href=True):
            href = a["href"]

            if href.startswith("/"):
                href = "https://www.20minutos.es" + href

            if not href.startswith("https://www.20minutos.es/"):
                continue

            if href.endswith(".html") or re.search(r"_\d+\.html$", href):
                enlaces.append(href)

        return list(dict.fromkeys(enlaces))

    except Exception as e:
        print(f"[ERROR tema] {url}: {e}")
        return []

def extraer_jsonld(soup):
    titulo = None
    fecha = None

    for script in soup.find_all("script", type="application/ld+json"):
        contenido = script.string or script.get_text()
        if not contenido:
            continue

        try:
            data = json.loads(contenido)
        except:
            continue

        items = data if isinstance(data, list) else [data]

        for item in items:
            if not isinstance(item, dict):
                continue

            if not titulo:
                titulo = item.get("headline") or item.get("name")

            if not fecha:
                fecha = item.get("datePublished") or item.get("dateCreated") or item.get("uploadDate")

            if titulo and fecha:
                return normalizar_texto(titulo), normalizar_texto(fecha)

    return titulo, fecha

def extraer_texto_noticia(soup):
    parrafos = []

    for p in soup.find_all("p"):
        txt = normalizar_texto(p.get_text(" ", strip=True))

        if len(txt) < 40:
            continue

        basura = [
            "whatsapp", "facebook", "twitter", "telegram",
            "copiar enlace", "copiar url", "publicidad"
        ]

        if txt.lower() in basura:
            continue

        parrafos.append(txt)

    texto = " ".join(parrafos)
    return normalizar_texto(texto)

def extraer_articulo(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        titulo_jsonld, fecha_jsonld = extraer_jsonld(soup)

        titulo = titulo_jsonld
        if not titulo:
            h1 = soup.find("h1")
            if h1:
                titulo = normalizar_texto(h1.get_text(" ", strip=True))

        fecha_visible = None
        time_tag = soup.find("time")
        if time_tag:
            fecha_visible = normalizar_texto(time_tag.get_text(" ", strip=True))

        fecha_publicacion = None
        fecha_dt = None

        if fecha_visible:
            fecha_dt = parsear_fecha_visible_20minutos(fecha_visible)
            fecha_publicacion = fecha_visible

        if fecha_dt is None and fecha_jsonld:
            fecha_dt = parsear_fecha_iso(fecha_jsonld)
            fecha_publicacion = fecha_jsonld

        texto = extraer_texto_noticia(soup)

        if not titulo or not texto:
            return None

        return {
            "Periodico": "20minutos",
            "Fecha": fecha_publicacion,
            "Fecha_dt": fecha_dt,
            "Titulo": titulo,
            "Texto": texto,
            "URL": url
        }

    except Exception as e:
        print(f"[ERROR noticia] {url}: {e}")
        return None

# Coger los enlances desde todas las webs
urls = []
for base in BASES_TEMA:
    paginas = construir_paginas(base, max_paginas=8)
    for p in paginas:
        print("Leyendo página de tema:", p)
        encontrados = sacar_enlaces_desde_pagina_tema(p)
        print(" -> enlaces:", len(encontrados))
        urls.extend(encontrados)
        time.sleep(1)

urls = list(dict.fromkeys(urls))
print("\nURLs únicas encontradas:", len(urls))

#Abrir noticias
#Coger los datos reals y extraerlos
#texto

filas = []
for i, url in enumerate(urls, start=1):
    print(f"[{i}/{len(urls)}] {url}")
    dato = extraer_articulo(url)
    if dato:
        filas.append(dato)
    time.sleep(0.8)

df = pd.DataFrame(filas)

if df.empty:
    raise ValueError("No se pudo extraer ningún artículo.")

# Filtrar por Ormuz
# Filtrar rango de fechas

df = df[df["Titulo"].fillna("").str.strip() != ""].copy()
df = df[df["Titulo"].apply(contiene_ormuz)].copy()
df = df[df["Fecha_dt"].apply(esta_en_rango)].copy()


df = df.drop_duplicates(subset=["URL"]).sort_values("Fecha_dt").reset_index(drop=True)

# Columnas finales
df_final = df[["Periodico", "Fecha", "Titulo", "Texto", "URL"]].copy()

# columnas minúsculas
df_final.columns = ["periodico", "fecha", "titulo", "texto", "url"]

# CSV
nombre_csv = "data_20minutos.csv"
df_final.to_csv(nombre_csv, index=False, encoding="utf-8")

print("\nTotal encontrados:", len(df_final))
display(df_final.head())
print("\nCSV guardado como:", nombre_csv)

Leyendo página de tema: https://www.20minutos.es/tags/temas/estrecho-ormuz.html
 -> enlaces: 45
Leyendo página de tema: https://www.20minutos.es/tags/temas/estrecho-ormuz.html/2/
 -> enlaces: 47
Leyendo página de tema: https://www.20minutos.es/tags/temas/estrecho-ormuz.html/3/
[ERROR tema] https://www.20minutos.es/tags/temas/estrecho-ormuz.html/3/: 404 Client Error: Not Found for url: https://www.20minutos.es/tags/temas/estrecho-ormuz.html/3/
 -> enlaces: 0
Leyendo página de tema: https://www.20minutos.es/tags/temas/estrecho-ormuz.html/4/
[ERROR tema] https://www.20minutos.es/tags/temas/estrecho-ormuz.html/4/: 404 Client Error: Not Found for url: https://www.20minutos.es/tags/temas/estrecho-ormuz.html/4/
 -> enlaces: 0
Leyendo página de tema: https://www.20minutos.es/tags/temas/estrecho-ormuz.html/5/
[ERROR tema] https://www.20minutos.es/tags/temas/estrecho-ormuz.html/5/: 404 Client Error: Not Found for url: https://www.20minutos.es/tags/temas/estrecho-ormuz.html/5/
 -> enlaces: 0
Leye

,periodico,fecha,titulo,texto,url
0,20minutos,2026-03-02T07:43:42+0100,Guerra en Irán y última hora del ataque de Isr...,Piden nueve años para Álvaro Aguado por agresi...,https://www.20minutos.es/internacional/guerra-...
1,20minutos,2026-03-07T15:46:58+0100,La Guardia Revolucionaria de Irán ataca dos pe...,Piden nueve años para Álvaro Aguado por agresi...,https://www.20minutos.es/internacional/guardia...
2,20minutos,2026-03-12T05:00:28+0100,"Minas navales, el ""arma que espera"" con la que...",Piden nueve años para Álvaro Aguado por agresi...,https://www.20minutos.es/internacional/minas-n...
3,20minutos,2026-03-14T08:26:20+0100,EEUU presiona a sus aliados y a China para rea...,Piden nueve años para Álvaro Aguado por agresi...,https://www.20minutos.es/internacional/guerra-...
4,20minutos,2026-03-15T18:09:01+0100,"Reino Unido dice ahora que analiza ""intensamen...",Piden nueve años para Álvaro Aguado por agresi...,https://www.20minutos.es/internacional/reino-u...



CSV guardado como: data_20minutos.csv


In [21]:
from google.colab import files
files.download("data_20minutos.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>